# Predictive Analytics: Academic Performance Classification

**Objective:** Develop a robust binary classification model using PySpark's MLlib. This pipeline ingests behavioral metrics (study habits, attendance, sleep patterns) to predict successful outcomes (Pass/Fail) using Logistic Regression.

**Key Techniques:** Logistic Regression, binary classification evaluation, and predictive modeling.

In [1]:
import findspark
findspark.init()
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("Assignment2").getOrCreate()

df = spark.read.csv("student_performance_large.csv", header=True, inferSchema=True)
df.show(5)

+-----------+----------+-----------+-----------+------+
|study_hours|attendance|assignments|sleep_hours|result|
+-----------+----------+-----------+-----------+------+
|          2|        41|          5|          5|     0|
|          4|        48|          2|          8|     0|
|          2|        77|          7|          4|     1|
|          1|        45|          4|          5|     0|
|          9|        78|          1|          8|     1|
+-----------+----------+-----------+-----------+------+
only showing top 5 rows


### Count the total number of students.

In [2]:
total_students = df.count()
print(f"Total number of students: {total_students}")

Total number of students: 150


### Calculate the average study_hours for all students.

In [3]:
df.agg(avg("study_hours").alias("Avg_Study_Hours")).show()

+---------------+
|Avg_Study_Hours|
+---------------+
|           5.18|
+---------------+



### How many students studied more than 5 hours?

In [4]:
studied_more_than_5 = df.filter(col("study_hours") > 5).count()
print(f"Students who studied more than 5 hours: {studied_more_than_5}")

Students who studied more than 5 hours: 67


### How many students have attendance less than 60?

In [5]:
attendance_less_60 = df.filter(col("attendance") < 60).count()
print(f"Students with attendance less than 60: {attendance_less_60}")

Students with attendance less than 60: 51


### How many students passed and how many failed?

In [6]:
print("Passed = 1, Failed = 0")
df.groupBy("result").count().show()

Passed = 1, Failed = 0
+------+-----+
|result|count|
+------+-----+
|     1|  126|
|     0|   24|
+------+-----+



### Build a model to predict the result (Pass = 1, Fail = 0) using: study_hours, attendance, assignments, sleep_hours

In [7]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Combine the required features into a single vector column
assembler = VectorAssembler(
    inputCols=["study_hours", "attendance", "assignments", "sleep_hours"],
    outputCol="features"
)

df_features = assembler.transform(df)

# Split the data into 80% for training and 20% for testing
train_data, test_data = df_features.randomSplit([0.8, 0.2], seed=42)

# Create the model and train it on train_data
lr = LogisticRegression(featuresCol="features", labelCol="result")
model = lr.fit(train_data)

print("Model successfully built and trained on 80% of the data!")

Model successfully built and trained on 80% of the data!


### Show the predictions: result, prediction

In [8]:
# Generate predictions using test_data
predictions = model.transform(test_data)

# Show actual result vs prediction
print("Predictions on the test data:")
predictions.select("result", "prediction").show(20)

Predictions on the test data:
+------+----------+
|result|prediction|
+------+----------+
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
|     0|       0.0|
|     0|       0.0|
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
+------+----------+
only showing top 20 rows


### Count how many predictions are correct.

In [9]:
correct_predictions = predictions.filter(col("result") == col("prediction")).count()
total_predictions = predictions.count()

accuracy = (correct_predictions / total_predictions) * 100

print(f"The model made {correct_predictions} correct predictions out of {total_predictions} test cases.")
print(f"Model Accuracy: {accuracy:.2f}%")

The model made 24 correct predictions out of 24 test cases.
Model Accuracy: 100.00%
